In [1]:
import geopandas as gpd

# read in the greenland boundary, ice sheet and ArcticDEM mosaic index shapefiles
boundary = gpd.read_file('data/vectors/greenland/GRL_adm0.shp').to_crs(epsg=3413)
ice_sheet = gpd.read_file('data/vectors/greenland/GRE_IceSheet_IMBIE2_v1.shp').to_crs(epsg=3413)
index = gpd.read_file('data/vectors/greenland/ArcticDEM_Mosaic_Index_v4_1_2m.shp').to_crs(epsg=3413)

# find area of Greenland surrounding the ice sheet
periphery = boundary.overlay(ice_sheet, how='difference')

# locate tiles from the index which correspond to the periphery
gland_index = index[index.intersects(periphery.union_all())]

# extract corresponding urls for tiles of interest 
tiles = list(gland_index['fileurl'])



/home/jamiemac/miniconda3/envs/geospatial/lib/python3.14/site-packages/geopandas/tools/overlay.py:358: UserWarning: `keep_geom_type=True` in overlay resulted in 10542 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)


In [2]:
short = [tiles[0]]
short

['https://data.pgc.umn.edu/elev/dem/setsm/ArcticDEM/mosaic/v4.1/2m/16_45/16_45_1_2_2m_v4.1.tar.gz']

In [3]:
from pathlib import Path
import subprocess

# specify a folder in which to save downloaded ArcticDEM tiles
folder = Path('data/rasters/greenland/tars')
folder.mkdir(parents=True, exist_ok=True)

# function which downloads the tiles of interest into the previously specified folder, only if they don't already exist
def get_data(tiles):
    for tile in tiles:
        url = tile
        cmd = [
            "wget",
            "-N",
            "-P", folder,
            url
        ]
        subprocess.run(cmd, check=True)

In [4]:
get_data(short)

--2026-06-30 13:33:05--  https://data.pgc.umn.edu/elev/dem/setsm/ArcticDEM/mosaic/v4.1/2m/16_45/16_45_1_2_2m_v4.1.tar.gz
Resolving data.pgc.umn.edu (data.pgc.umn.edu)... 134.84.66.103
Connecting to data.pgc.umn.edu (data.pgc.umn.edu)|134.84.66.103|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1840070173 (1.7G) [application/x-gzip]
Saving to: ‘data/rasters/greenland/tars/16_45_1_2_2m_v4.1.tar.gz’

     0K .......... .......... .......... .......... ..........  0% 4.86M 6m1s
    50K .......... .......... .......... .......... ..........  0%  465K 35m13s
   100K .......... .......... .......... .......... ..........  0% 4.07M 25m52s
   150K .......... .......... .......... .......... ..........  0%  506K 34m11s
   200K .......... .......... .......... .......... ..........  0% 3.95M 28m50s
   250K .......... .......... .......... .......... ..........  0%  525K 33m32s
   300K .......... .......... .......... .......... ..........  0% 4.42M 29m41s
   350K ......

In [ ]:
with tarfile.open('16_45_1_2_2m_v4.1.tar.gz', 'r') as tar:
...     members = tar.getmembers()
...     tar.extract(members[0])
